# Lesson X1: RL Debugging - Practitioner Toolkit

## Introduction

Every notebook so far assumed the algorithm was implemented correctly and just needed training. In practice, most RL debugging time goes to a different question: **is anything actually wrong, and how would you tell?** RL bugs are unusually hard to diagnose because a broken agent and a *correctly implemented but not-yet-converged* agent often look identical — both start out failing.

1. **Common failure modes**: a catalogue, with a concrete from-scratch demonstration of the sneakiest one — reward hacking
2. **Logging and visualization**: TensorBoard and Weights & Biases, wired into an SB3 training run
3. **Reproducibility and hyperparameter sensitivity**: quantify how much of your result is the algorithm and how much is the seed

## Setup

In [ ]:
import os
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import defaultdict

np.random.seed(42)
os.environ['WANDB_MODE'] = 'offline'  # no W&B account needed to run this notebook;
                                       # remove this line and `wandb login` for real cloud logging

## Part 1: Common RL Failure Modes

- **Reward hacking / specification gaming**: the agent finds a way to maximize reward that technically satisfies the reward function but not the actual intent — demonstrated below
- **Sparse-reward stalls**: no reward signal ever fires, so there's nothing to learn from (13B's whole premise)
- **Overestimation bias**: value-based methods bootstrapping through `max` systematically overestimate, especially offline (14A/14B)
- **Seed sensitivity**: identical hyperparameters, wildly different outcomes across random seeds (Part 3 below quantifies this directly)
- **The deadly triad**: function approximation + bootstrapping + off-policy learning together can diverge, even though any two alone are usually fine
- **Distribution shift**: a policy trained on one state distribution behaves unpredictably once it drifts outside it (15A/15B, offline RL generally)
- **Non-stationarity**: in multi-agent settings, the "environment" changes as other agents learn (12A/12B)

### Reward Hacking, Concretely

A GridWorld with a **badly specified** reward: `+1` for every step spent inside a 3-cell region near the goal, with no bonus for actually reaching the goal cell and no episode termination. Compare against a **properly specified** reward: `+10` once, on actually reaching the goal, which then ends the episode.

In [ ]:
SIZE = 6
GOAL_ZONE = {(5, 5), (5, 4), (4, 5)}
TRUE_GOAL = (5, 5)
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


def step(pos, action, hacked_reward):
    di, dj = DELTAS[action]
    next_pos = (max(0, min(SIZE - 1, pos[0] + di)), max(0, min(SIZE - 1, pos[1] + dj)))
    if hacked_reward:
        reward = 1.0 if next_pos in GOAL_ZONE else -0.1
        done = False  # never terminates -- nothing punishes loitering
    else:
        reward = 10.0 if next_pos == TRUE_GOAL else -0.1
        done = next_pos == TRUE_GOAL
    return next_pos, reward, done


def train(hacked_reward, n_episodes=300, max_steps=30, alpha=0.3, gamma=0.95, epsilon=0.15):
    Q = defaultdict(lambda: np.zeros(4))
    for _ in range(n_episodes):
        pos = (0, 0)
        for _ in range(max_steps):
            action = np.random.randint(4) if np.random.rand() < epsilon else int(np.argmax(Q[pos]))
            next_pos, reward, done = step(pos, action, hacked_reward)
            Q[pos][action] += alpha * (reward + gamma * np.max(Q[next_pos]) - Q[pos][action])
            pos = next_pos
            if done:
                break
    return Q


def rollout(Q, hacked_reward, max_steps=30):
    pos = (0, 0)
    trajectory = [pos]
    for _ in range(max_steps):
        action = int(np.argmax(Q[pos]))
        pos, _, done = step(pos, action, hacked_reward)
        trajectory.append(pos)
        if done:
            break
    return trajectory


Q_hacked = train(hacked_reward=True)
Q_proper = train(hacked_reward=False)
trajectory_hacked = rollout(Q_hacked, hacked_reward=True)
trajectory_proper = rollout(Q_proper, hacked_reward=False)

print(f"Hacked-reward policy: ever reaches the true goal {TRUE_GOAL}? {TRUE_GOAL in trajectory_hacked}")
print(f"Hacked-reward policy: episode length = {len(trajectory_hacked) - 1} (maxed out -- never terminates)")
print(f"Hacked-reward policy: settles at {trajectory_hacked[-1]} and stays there, farming +1/step forever")
print()
print(f"Properly-specified policy: episode length = {len(trajectory_proper) - 1}")
print(f"Properly-specified policy: final position = {trajectory_proper[-1]} (the actual goal)")
print()
print("Both policies were trained with the identical algorithm on the identical grid -- only the")
print("reward SPECIFICATION differs. Training curves and losses would look equally 'healthy' for")
print("both; only checking what the policy actually DOES reveals the hack.")

## Part 2: Logging and Visualization - TensorBoard and Weights & Biases

Both tools solve the same core problem: RL training curves are noisy, multi-metric, and span thousands of updates — impossible to debug from `print()` statements alone. SB3 writes TensorBoard-compatible logs natively (`tensorboard_log=...`); W&B's `sync_tensorboard=True` mirrors the exact same event stream into a shareable, cloud-hosted dashboard with zero extra logging code.

In [ ]:
import wandb
from stable_baselines3 import PPO

wandb_run = wandb.init(project='rl-debugging-demo', mode='offline', sync_tensorboard=True,
                        dir='/tmp/wandb_demo')

env = gym.make('CartPole-v1')
model = PPO('MlpPolicy', env, verbose=0, device='cpu',
            tensorboard_log=f'/tmp/tb_logs/{wandb_run.id}')
model.learn(total_timesteps=8000)
wandb_run.finish()

print(f"TensorBoard logs: /tmp/tb_logs/{wandb_run.id}  (view with: tensorboard --logdir /tmp/tb_logs)")
print(f"W&B offline run saved locally: {wandb_run.dir}")
print("To sync to the cloud dashboard: `wandb login` once, then `wandb sync <run dir>`,")
print("or drop WANDB_MODE=offline entirely to stream live during training.")

## Part 3: Reproducibility and Hyperparameter Sensitivity

The uncomfortable truth about RL benchmarks: a single training run's score is one sample from a noisy distribution, not a repeatable measurement. We train PPO on CartPole across 3 learning rates x 3 seeds each (9 runs total) to separate two different sources of variation: **which hyperparameter setting is best on average**, and **how much any single run can differ from that average purely from its random seed**.

In [ ]:
def evaluate(model, env, n_episodes=10):
    lengths = []
    for ep in range(n_episodes):
        state, _ = env.reset(seed=900 + ep)
        for t in range(500):
            action, _ = model.predict(state, deterministic=True)
            state, reward, terminated, truncated, _ = env.step(action)
            if terminated or truncated:
                break
        lengths.append(t + 1)
    return np.mean(lengths)


learning_rates = [1e-4, 1e-3, 1e-2]
results = {}
for lr in learning_rates:
    scores = []
    for seed in range(3):
        lr_env = gym.make('CartPole-v1')
        lr_model = PPO('MlpPolicy', lr_env, learning_rate=lr, verbose=0, device='cpu', seed=seed)
        lr_model.learn(total_timesteps=8000)
        scores.append(evaluate(lr_model, lr_env))
    results[lr] = scores
    print(f"lr={lr}: per-seed scores={[round(s, 0) for s in scores]}, "
          f"mean={np.mean(scores):.0f}, std={np.std(scores):.0f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
xs = range(len(learning_rates))
means = [np.mean(results[lr]) for lr in learning_rates]
stds = [np.std(results[lr]) for lr in learning_rates]
for i, lr in enumerate(learning_rates):
    ax.scatter([i] * len(results[lr]), results[lr], color='C0', alpha=0.6, label='individual seeds' if i == 0 else None)
ax.errorbar(xs, means, yerr=stds, fmt='o-', color='C1', capsize=6, label='mean +/- std across seeds')
ax.set_xticks(list(xs))
ax.set_xticklabels([f'lr={lr}' for lr in learning_rates])
ax.set_ylabel('Mean episode length (10 eval episodes)')
ax.set_title('Hyperparameter sensitivity AND seed variance, same 9 runs')
ax.legend()
plt.tight_layout()
plt.show()

print("Reading this plot correctly means asking two separate questions: (1) which lr's mean is")
print("highest, and (2) is that mean's gap over the others bigger than the seed-to-seed spread")
print("WITHIN each setting? If not, you don't have enough seeds to trust the comparison yet --")
print("a single-seed 'my hyperparameter is better' claim is exactly the trap this section exists to avoid.")

## Key Takeaways

1. **Reward hacking is invisible in the training curve** — loss and reward-per-step can look perfectly healthy while the policy exploits a specification gap; the only way to catch it is inspecting what the policy actually *does*
2. **TensorBoard and W&B log the same underlying event stream** — `sync_tensorboard=True` gets both for the cost of one
3. **A single training run is a sample, not a measurement** — report mean and spread across multiple seeds, not one number
4. **Hyperparameter comparisons need enough seeds to separate real effects from seed noise** — if the gap between settings is smaller than the within-setting spread, you haven't shown anything yet
5. This notebook's own catalogue (reward hacking, sparse reward, overestimation, seed sensitivity, the deadly triad, distribution shift, non-stationarity) each got a dedicated, hands-on demonstration somewhere earlier in this curriculum — X1 is the practitioner's index back into all of them